# Phase 1 — Data Integration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)

RAW_DIR = Path('../data/raw')
PROC_DIR = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

dataset = {
    'customers': 'olist_customers_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'payments': 'olist_order_payments_dataset.csv',
    'reviews': 'olist_order_reviews_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'category_translation': 'product_category_name_translation.csv',
}

dfs = {}
for name, fname in dataset.items():
    path = RAW_DIR / fname
    dfs[name] = pd.read_csv(path)
    print(f"{name:22s} shape={dfs[name].shape}")

print()

for name, df in dfs.items():
    print(f"--- {name} ---")
    print(df.dtypes)
    print()



customers              shape=(99441, 5)
orders                 shape=(99441, 8)
order_items            shape=(112650, 7)
payments               shape=(103886, 5)
reviews                shape=(99224, 7)
products               shape=(32951, 9)
sellers                shape=(3095, 4)
geolocation            shape=(1000163, 5)
category_translation   shape=(71, 2)

--- customers ---
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

--- orders ---
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

--- order_items ---
order_id                   str
order_item_id            int64
product_id                 s

### Primary Keys

In [2]:
def check_primary_key(df, cols, name):
    """A column (or column list) is a valid PK if it's unique and non-null."""
    n_rows = len(df)
    n_unique = df.drop_duplicates(subset=cols).shape[0]
    n_nulls = df[cols].isna().any(axis=1).sum()
    is_valid = (n_unique == n_rows) and (n_nulls == 0)
    status = 'VALID PK' if is_valid else 'NOT a valid PK'
    print(f"{name:28s} cols={cols}  unique={n_unique}/{n_rows}  nulls={n_nulls}  -> {status}")
    return is_valid

# Candidate primary keys per table
check_primary_key(dfs['customers'], ['customer_id'], 'customers')
check_primary_key(dfs['customers'], ['customer_unique_id'], 'customers (unique_id)')  
check_primary_key(dfs['orders'], ['order_id'], 'orders')
check_primary_key(dfs['order_items'], ['order_id'], 'order_items (order_id alone)')  
check_primary_key(dfs['order_items'], ['order_id', 'order_item_id'], 'order_items (composite)')
check_primary_key(dfs['payments'], ['order_id', 'payment_sequential'], 'payments (composite)')
check_primary_key(dfs['reviews'], ['review_id'],'reviews (review_id alone)')
check_primary_key(dfs['reviews'], ['review_id', 'order_id'],'reviews (composite)')
check_primary_key(dfs['products'], ['product_id'], 'products')
check_primary_key(dfs['sellers'], ['seller_id'], 'sellers')


customers                    cols=['customer_id']  unique=99441/99441  nulls=0  -> VALID PK
customers (unique_id)        cols=['customer_unique_id']  unique=96096/99441  nulls=0  -> NOT a valid PK
orders                       cols=['order_id']  unique=99441/99441  nulls=0  -> VALID PK
order_items (order_id alone) cols=['order_id']  unique=98666/112650  nulls=0  -> NOT a valid PK
order_items (composite)      cols=['order_id', 'order_item_id']  unique=112650/112650  nulls=0  -> VALID PK
payments (composite)         cols=['order_id', 'payment_sequential']  unique=103886/103886  nulls=0  -> VALID PK
reviews (review_id alone)    cols=['review_id']  unique=98410/99224  nulls=0  -> NOT a valid PK
reviews (composite)          cols=['review_id', 'order_id']  unique=99224/99224  nulls=0  -> VALID PK
products                     cols=['product_id']  unique=32951/32951  nulls=0  -> VALID PK
sellers                      cols=['seller_id']  unique=3095/3095  nulls=0  -> VALID PK


np.True_

### Foreign Keys 

In [3]:
def check_foreign_key(child_df, child_col, parent_df, parent_col, child_name, parent_name):
    """A valid FK: every non-null child value exists in the parent's key column
    (no orphan references)."""
    child_vals = set(child_df[child_col].dropna().unique())
    parent_vals = set(parent_df[parent_col].dropna().unique())
    orphans = child_vals - parent_vals
    status = 'VALID FK (no orphans)' if len(orphans) == 0 else f'{len(orphans)} ORPHAN VALUES'
    print(f"{child_name}.{child_col:20s} -> {parent_name}.{parent_col:20s}  {status}")
    return orphans

# Candidate foreign key relationships
check_foreign_key(dfs['orders'], 'customer_id', dfs['customers'], 'customer_id', 'orders', 'customers')
check_foreign_key(dfs['order_items'], 'order_id', dfs['orders'], 'order_id', 'order_items', 'orders')
check_foreign_key(dfs['order_items'], 'product_id', dfs['products'], 'product_id', 'order_items', 'products')
check_foreign_key(dfs['order_items'], 'seller_id', dfs['sellers'], 'seller_id', 'order_items', 'sellers')
check_foreign_key(dfs['payments'], 'order_id', dfs['orders'], 'order_id', 'payments', 'orders')
check_foreign_key(dfs['reviews'], 'order_id', dfs['orders'], 'order_id', 'reviews', 'orders')
_ = check_foreign_key(dfs['products'], 'product_category_name', dfs['category_translation'],
                       'product_category_name', 'products', 'category_translation')


orders.customer_id          -> customers.customer_id           VALID FK (no orphans)
order_items.order_id             -> orders.order_id              VALID FK (no orphans)
order_items.product_id           -> products.product_id            VALID FK (no orphans)
order_items.seller_id            -> sellers.seller_id             VALID FK (no orphans)
payments.order_id             -> orders.order_id              VALID FK (no orphans)
reviews.order_id             -> orders.order_id              VALID FK (no orphans)
products.product_category_name -> category_translation.product_category_name  2 ORPHAN VALUES


### One-to-Many Relationships

In [4]:
def check_one_to_many(parent_df, parent_key, child_df, child_key, parent_name, child_name):
    """Groups the child table by its FK and reports how many child rows map
    to each parent — confirming (or disproving) a one-to-many relationship."""
    counts = child_df.groupby(child_key).size()
    n_parents_with_children = counts.shape[0]
    n_one_to_many = (counts > 1).sum()
    max_children = counts.max()
    avg_children = counts.mean()
    relationship = 'ONE-TO-MANY' if n_one_to_many > 0 else 'ONE-TO-ONE (not one-to-many)'
    print(f"{parent_name} -> {child_name:14s} "
          f"parents_with_children={n_parents_with_children:6d}  "
          f"parents_with_multiple_children={n_one_to_many:6d}  "
          f"max_children_per_parent={max_children:3d}  "
          f"avg={avg_children:.2f}  -> {relationship}")
    return counts

# customers -> orders: does one customer_unique_id place multiple orders?
_ = check_one_to_many(dfs['customers'], 'customer_unique_id', dfs['orders'].merge(
        dfs['customers'][['customer_id', 'customer_unique_id']], on='customer_id'),
        'customer_unique_id', 'customers', 'orders')

# orders -> order_items: does one order contain multiple line items?
_ = check_one_to_many(dfs['orders'], 'order_id', dfs['order_items'], 'order_id',
                       'orders', 'order_items')

# orders -> payments: does one order have multiple payment installments?
_ = check_one_to_many(dfs['orders'], 'order_id', dfs['payments'], 'order_id',
                       'orders', 'payments')

# products -> order_items: is one product sold in multiple order line items?
_ = check_one_to_many(dfs['products'], 'product_id', dfs['order_items'], 'product_id',
                       'products', 'order_items')

# sellers -> order_items: does one seller appear in multiple order line items?
_ = check_one_to_many(dfs['sellers'], 'seller_id', dfs['order_items'], 'seller_id',
                       'sellers', 'order_items')


customers -> orders         parents_with_children= 96096  parents_with_multiple_children=  2997  max_children_per_parent= 17  avg=1.03  -> ONE-TO-MANY
orders -> order_items    parents_with_children= 98666  parents_with_multiple_children=  9803  max_children_per_parent= 21  avg=1.14  -> ONE-TO-MANY
orders -> payments       parents_with_children= 99440  parents_with_multiple_children=  2961  max_children_per_parent= 29  avg=1.04  -> ONE-TO-MANY
products -> order_items    parents_with_children= 32951  parents_with_multiple_children= 14834  max_children_per_parent=527  avg=3.42  -> ONE-TO-MANY
sellers -> order_items    parents_with_children=  3095  parents_with_multiple_children=  2586  max_children_per_parent=2033  avg=36.40  -> ONE-TO-MANY


### Data joining techniques

Merge order:
1. `orders` + `customers` (on `customer_id`)
2. + `order_items` (on `order_id`) — row count grows here (1 order → many items)
3. + `products` (on `product_id`)
4. + `category_translation` (on `product_category_name`)
5. + `sellers` (on `seller_id`)
6. + `payments` (aggregated to 1 row/order first, then merged on `order_id`)
7. + `reviews` (deduplicated to 1 row/order, then merged on `order_id`)

Row counts are printed after each step to catch unexpected fan-outs.

In [5]:
# 1. orders + customers
df = dfs['orders'].merge(dfs['customers'], on='customer_id', how='left')
print('after customers:', df.shape)

# 2. + order_items
df = df.merge(dfs['order_items'], on='order_id', how='inner')
print('after order_items:', df.shape)

# 3. + products
df = df.merge(dfs['products'], on='product_id', how='left')
print('after products:', df.shape)

# 4. + category translation
df = df.merge(dfs['category_translation'], on='product_category_name', how='left')
print('after category_translation:', df.shape)

# 5. + sellers
df = df.merge(dfs['sellers'], on='seller_id', how='left')
print('after sellers:', df.shape)


after customers: (99441, 12)
after order_items: (112650, 18)
after products: (112650, 26)
after category_translation: (112650, 27)
after sellers: (112650, 30)


In [6]:
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english', 'seller_zip_code_prefix',
       'seller_city', 'seller_state'],
      dtype='str')

In [7]:
# 6. payments — aggregate to 1 row per order first (avoids fanning out order_items)
payments_agg = dfs['payments'].groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
    payment_type=('payment_type', lambda x: x.mode().iat[0] if not x.mode().empty else np.nan),
    n_payment_methods=('payment_type', 'nunique'),
).reset_index()

df = df.merge(payments_agg, on='order_id', how='left')
print('after payments:', df.shape)

# 7. reviews — occasionally >1 review per order; keep the most recent
reviews_sorted = dfs['reviews'].sort_values('review_creation_date')
reviews_dedup = reviews_sorted.drop_duplicates(subset='order_id', keep='last')
df = df.merge(
    reviews_dedup[['order_id', 'review_score', 'review_comment_message',
                    'review_creation_date', 'review_answer_timestamp']],
    on='order_id', how='left'
)
print('after reviews:', df.shape)


after payments: (112650, 34)
after reviews: (112650, 38)


In [8]:
print('Final merged shape:', df.shape)
print('Unique customers (customer_unique_id):', df['customer_unique_id'].nunique())
print('Unique orders:', df['order_id'].nunique())
df.head()


Final merged shape: (112650, 38)
Unique customers (customer_unique_id): 95420
Unique orders: 98666


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value,payment_installments,payment_type,n_payment_methods,review_score,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,02-10-2017 10:56,02-10-2017 11:07,04-10-2017 19:55,10-10-2017 21:25,18-10-2017 00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares,9350,maua,SP,38.71,1.0,voucher,2.0,4.0,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,24-07-2018 20:41,26-07-2018 03:24,26-07-2018 14:31,07-08-2018 15:27,13-08-2018 00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery,31570,belo horizonte,SP,141.46,1.0,boleto,1.0,4.0,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,08-08-2018 08:38,08-08-2018 08:55,08-08-2018 13:50,17-08-2018 18:06,04-09-2018 00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto,14840,guariba,SP,179.12,3.0,credit_card,1.0,5.0,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,18-11-2017 19:28,18-11-2017 19:45,22-11-2017 13:39,02-12-2017 00:28,15-12-2017 00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0,pet_shop,31842,belo horizonte,MG,72.20,1.0,credit_card,1.0,5.0,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,13-02-2018 21:18,13-02-2018 22:20,14-02-2018 19:46,16-02-2018 18:17,26-02-2018 00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0,stationery,8752,mogi das cruzes,SP,28.62,1.0,credit_card,1.0,5.0,NaN,2018-02-17 00:00:00,2018-02-18 13:02:51


In [9]:
# Save the RAW merge — no cleaning applied yet. Phase 2 picks up from here.
out_path = PROC_DIR / 'olist_merged_raw.csv'
df.to_csv(out_path, index=False)
print(f'Saved to {out_path}  ({df.shape[0]:,} rows, {df.shape[1]} cols)')


Saved to ..\data\processed\olist_merged_raw.csv  (112,650 rows, 38 cols)
